# Frameworks de agentes de voz: del sándwich al voice-to-voice

**Lección 4 · Clase 5.3** — la lección 3 terminó con tres problemas que no eran de implementación sino de **arquitectura**: se perdía la prosodia del usuario, interrumpir era plomería difícil, y los turnos eran rígidos.

Los tres tienen la misma causa. En el sándwich, el audio se convierte en **texto** antes de que el modelo lo vea, y el texto no tiene tono. Los modelos *voice-to-voice* atacan eso por la raíz:

```
SÁNDWICH (lección 3)
  audio ─▶ STT ─▶ texto ─▶ LLM ─▶ texto ─▶ TTS ─▶ audio      3 modelos, 3 saltos

VOICE-TO-VOICE (esta lección)
  audio ────────────▶ LLM ────────────▶ audio                 1 modelo, 0 saltos
```

Un modelo como `gpt-realtime-2.1` recibe el audio y **emite audio**, en el mismo modelo y en streaming continuo. Oye que estás dudando, oye que estás molesto, y nota cuando lo interrumpes porque nunca dejó de escuchar.

Esta lección no es un notebook de ejecución: los agentes de voz necesitan micrófono, parlantes y un bucle de audio, así que viven en **scripts**. El notebook es la guía — explica el salto, mide la diferencia que importa, y compara **tres frameworks** construyendo *el mismo agente*.

## El mismo agente, cuatro veces

Para que la comparación sea entre frameworks y no entre demos, los cuatro programas resuelven el mismo caso: **Luis**, que atiende el teléfono de una ferretería y consulta el stock con una herramienta. El dominio vive en [`tienda.py`](tienda.py) y los cuatro lo importan.

| Archivo | Framework | Qué representa |
|---|---|---|
| [`nivel0_websocket_crudo.py`](nivel0_websocket_crudo.py) | ninguno | La API a pelo. Está para responder "¿qué hace el framework por mí?" |
| [`agente_openai.py`](agente_openai.py) | **OpenAI Agents SDK** | Cerrado, del mismo proveedor del modelo. Los primitivos de agentes que ya conoces. |
| [`agente_pipecat.py`](agente_pipecat.py) | **Pipecat** (Daily) | Open source. El pipeline explícito, cada pieza intercambiable. |
| [`agente_elevenlabs.py`](agente_elevenlabs.py) | **ElevenLabs Agents** | Plataforma. El agente no vive en tu código. |

Y una pieza compartida más: [`salida_fluida.py`](salida_fluida.py), el reproductor con *jitter buffer* que los cuatro usan para que el audio no salga a saltos. Más abajo se explica por qué hace falta.

In [ ]:
# Esta lección usa el entorno uv del README. NO corre en Colab: necesita micrófono local.
from dotenv import load_dotenv
import os
import subprocess
import sys

load_dotenv()

if not os.environ.get("ELEVENLABS_API_KEY") and os.environ.get("ELEVEN_API_KEY"):
    os.environ["ELEVENLABS_API_KEY"] = os.environ["ELEVEN_API_KEY"]

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_ELEVEN = bool(os.environ.get("ELEVENLABS_API_KEY"))
print("OPENAI_API_KEY presente    :", HAY_OPENAI)
print("ELEVENLABS_API_KEY presente:", HAY_ELEVEN, "(solo para agente_elevenlabs.py)")


def correr(*argumentos: str, timeout: int = 180) -> str:
    """Corre uno de los scripts de la lección y devuelve su salida."""
    proceso = subprocess.run(
        [sys.executable, *argumentos],
        capture_output=True, text=True, timeout=timeout,
        # Los scripts ya lo hacen por su cuenta; acá también, por si se corre desde el notebook.
        env={**os.environ, "NLTK_DISABLE_IMPORT_SECURITY": "1"},
    )
    salida = (proceso.stdout + proceso.stderr).strip()
    # Los logs de pipecat son ruidosos: dejamos solo lo nuestro
    return "\n".join(l for l in salida.splitlines() if not l.startswith("2026-"))

## Nivel 0: la API a pelo

Antes de comparar frameworks conviene ver qué pasa sin ninguno, porque si no, todo parece magia gratis. `nivel0_websocket_crudo.py` abre el WebSocket, manda un JSON de configuración, empuja bloques de audio en base64 y va interpretando eventos a mano.

Su modo `--smoke` hace el handshake y una pregunta de texto, sin tocar el micrófono. Sirve además como verificación de que tu llave y el modelo funcionan antes de pelearte con el audio.

In [ ]:
if HAY_OPENAI:
    print(correr("nivel0_websocket_crudo.py", "--smoke"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

Fíjate en lo que hubo que escribir a mano para eso (mira el archivo): la forma exacta del `session.update`, el decodificado de cada `delta` de audio en base64, y —el clásico— los **dos** mensajes que hay que mandar tras ejecutar una herramienta: primero el `function_call_output` y después un `response.create` para que el modelo siga hablando. Si te olvidas del segundo, el agente se queda mudo y no hay error que te lo diga.

> Un detalle que delata la edad del código que encuentres por internet: la forma del `session` **cambió**. El material original de esta clase usaba `input_audio_format: "pcm16"` y `turn_detection` en la raíz; hoy todo va anidado bajo `audio.input` / `audio.output` y el formato es un objeto (`{"type": "audio/pcm", "rate": 24000}`). Un ejemplo de 2024 no corre tal cual.

Eso —el protocolo, los formatos, el bucle de eventos, el orden de los mensajes— es exactamente lo que los tres frameworks resuelven.

## La diferencia que importa: tiempo hasta la primera sílaba

En la lección 3 medimos el sándwich completo: ~6 segundos, con el TTS aportando más de la mitad. Ahora la pregunta correcta no es cuánto demora *todo*, sino **cuánto tarda el usuario en oír la primera sílaba**, porque desde ahí ya está escuchando algo y la conversación se siente viva.

Medimos las dos arquitecturas partiendo del mismo punto —una pregunta en texto— hasta el primer byte de audio:

- **Sándwich**: el agente tiene que terminar de escribir su respuesta y solo entonces el TTS empieza a sintetizar.
- **Voice-to-voice**: el modelo empieza a emitir audio mientras todavía está decidiendo qué decir.

In [ ]:
import asyncio
import base64
import json
import time

PREGUNTA = "¿Tienen taladros? ¿Cuánto cuestan?"
MODELO_REALTIME = "gpt-realtime-2.1"


async def primer_byte_realtime() -> float:
    """Segundos desde la pregunta hasta el primer byte de audio (voice-to-voice)."""
    import websockets
    from tienda import INSTRUCCIONES

    url = f"wss://api.openai.com/v1/realtime?model={MODELO_REALTIME}"
    async with websockets.connect(
        url, additional_headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    ) as ws:
        await ws.recv()  # session.created
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                "instructions": INSTRUCCIONES,
                "output_modalities": ["audio"],
                "audio": {"output": {"format": {"type": "audio/pcm", "rate": 24000},
                                     "voice": "marin"}},
            },
        }))
        await ws.send(json.dumps({
            "type": "conversation.item.create",
            "item": {"type": "message", "role": "user",
                     "content": [{"type": "input_text", "text": PREGUNTA}]},
        }))

        inicio = time.perf_counter()
        await ws.send(json.dumps({"type": "response.create"}))
        async for crudo in ws:
            if json.loads(crudo)["type"] == "response.output_audio.delta":
                return time.perf_counter() - inicio
    return float("nan")


def primer_byte_sandwich() -> tuple[float, float]:
    """Segundos hasta el primer byte de audio con la cascada (y cuánto fue el LLM)."""
    from openai import OpenAI
    from tienda import INSTRUCCIONES

    cliente = OpenAI()
    inicio = time.perf_counter()

    respuesta = cliente.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "system", "content": INSTRUCCIONES},
                  {"role": "user", "content": PREGUNTA}],
    )
    texto = respuesta.choices[0].message.content
    tiempo_llm = time.perf_counter() - inicio

    # El TTS no puede empezar hasta tener el texto: ese es el costo de la arquitectura.
    with cliente.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts", voice="marin", input=texto, response_format="pcm",
    ) as flujo:
        for _ in flujo.iter_bytes(chunk_size=1024):
            break  # el primer trozo y listo
    return time.perf_counter() - inicio, tiempo_llm


if HAY_OPENAI:
    total_sandwich, tiempo_llm = primer_byte_sandwich()
    # `await` directo: el kernel de Jupyter ya tiene un event loop corriendo,
    # así que asyncio.run() daría RuntimeError. En un script sí se usa asyncio.run().
    tiempo_v2v = await primer_byte_realtime()

    print(f"{'arquitectura':<34} {'primera sílaba':>15}")
    print("-" * 52)
    print(f"{'Sándwich (LLM → TTS)':<34} {total_sandwich:>14.2f}s")
    print(f"{'   del cual, el LLM':<34} {tiempo_llm:>14.2f}s")
    print(f"{'Voice-to-voice (gpt-realtime-2.1)':<34} {tiempo_v2v:>14.2f}s")
    print("-" * 52)
    if tiempo_v2v < total_sandwich:
        print(f"→ el voice-to-voice habla {total_sandwich / tiempo_v2v:.1f}× antes")
else:
    print("⛔ Falta OPENAI_API_KEY.")

La diferencia no viene de que un modelo sea "más rápido": viene de que en el sándwich hay una **barrera**. El TTS no puede empezar hasta que el LLM haya terminado de escribir, porque necesita el texto. El modelo voice-to-voice no tiene esa barrera — está generando audio y contenido a la vez.

Y ojo con lo que esta medición **no** captura, que es la mitad del argumento: partimos de una pregunta en *texto*, así que le regalamos al sándwich la transcripción (que en la lección 3 costó ~1,3 s más). Con audio real, la brecha es más grande.

Tampoco captura lo que no es tiempo: que el modelo oiga tu tono, y que puedas interrumpirlo. Eso no se mide con un cronómetro, se prueba hablando — y para eso están los scripts.

## Framework 1 · OpenAI Agents SDK

El framework de agentes del mismo proveedor del modelo. Su gracia es la **continuidad**: `RealtimeAgent` usa los mismos primitivos que los agentes de texto que ya escribiste —instrucciones, `@function_tool`, handoffs, guardrails— y encima resuelve el protocolo.

```python
agente = RealtimeAgent(name="Luis", instructions=INSTRUCCIONES, tools=[consultar_stock_tool])

runner = RealtimeRunner(starting_agent=agente, config={
    "model_settings": {
        "model_name": "gpt-realtime-2.1",
        "audio": {
            "input": {"format": "pcm16",
                      "turn_detection": {"type": "semantic_vad", "interrupt_response": True}},
            "output": {"format": "pcm16", "voice": "marin"},
        },
    },
})
sesion = await runner.run()
```

Lo que **no** resuelve: el micrófono y los parlantes. Ese cableado con PyAudio es la mitad de [`agente_openai.py`](agente_openai.py) y explica por qué es el archivo más largo de los tres.

Vale detenerse en `semantic_vad`. Un VAD clásico corta el turno por **silencio**, así que si dudas a mitad de frase ("quiero… un taladro") te interrumpe. El semántico usa el contenido para decidir si terminaste. Es la clase de detalle que separa una demo de un producto, y acá es un string en la configuración.

In [ ]:
if HAY_OPENAI:
    print(correr("agente_openai.py", "--smoke"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

## Framework 2 · Pipecat (open source)

Pipecat, de Daily, modela la conversación como un **pipeline de frames**: el audio entra por un extremo, atraviesa una lista de procesadores y sale por el otro.

Ese modelo tiene una consecuencia que es el mejor argumento pedagógico de toda la lección: las dos arquitecturas de esta clase son **la misma lista, con dos elementos de diferencia**.

In [ ]:
if HAY_OPENAI:
    print(correr("agente_pipecat.py", "--check", "--modo", "cascada"))
    print()
    print(correr("agente_pipecat.py", "--check", "--modo", "realtime"))
else:
    print("⛔ Falta OPENAI_API_KEY.")

Ahí está, literal: en el modo `realtime` desaparecen `OpenAISTTService` y `OpenAITTSService`. El servicio de LLM **es** el modelo de voz. Y no es un detalle de este script — Pipecat trae sus propias plantillas `bot_cascade` y `bot_realtime`, o sea que la distinción que hicimos entre la lección 3 y la 4 es la misma que hace el framework.

Las dos cosas que Pipecat da y los otros no:

- **Cada pieza es intercambiable.** Cambiar el STT de OpenAI a Deepgram, o el TTS a ElevenLabs o Cartesia, es cambiar una línea. Si tu apuesta es no quedar amarrado a un proveedor, esto es el argumento.
- **VAD local.** El `SileroVADAnalyzer` corre en tu máquina: la detección de silencios no manda audio a ningún servidor. Para datos sensibles, importa.

Su costo es la superficie: es el framework con más conceptos que aprender (frames, procesadores, agregadores de contexto, workers) y el que más se mueve entre versiones.

> **Una trampa concreta que te vas a encontrar.** Pipecat importa `nltk`, que trae un hook de seguridad que bloquea cualquier módulo cuyo origen esté *dentro del directorio de trabajo*. Como nuestro `.venv` vive justamente ahí —la convención de `uv`— el import se cae con un mensaje bastante desorientador sobre `regex`. Se arregla con `NLTK_DISABLE_IMPORT_SECURITY=1`, y los scripts de esta lección ya lo hacen por su cuenta.

### Y una segunda trampa, esta audible: el audio sale a saltos

Al probar el agente de Pipecat con audífonos, el audio se oye **cortado**, con un chasquido casi continuo. No es la red y no es el modelo: es el `LocalAudioTransport`.

Su salida escribe al dispositivo con `stream.write()` **bloqueante**, entregando trozos de 40 ms uno tras otro. Eso deja al parlante casi sin cojín, y en un proceso que además corre VAD, STT, LLM y TTS en el mismo event loop, el productor se retrasa constantemente unos milisegundos. Cada retraso es un microcorte.

Se puede medir sin oído. PyAudio expone la señal exacta: `stream.write(..., exception_on_underflow=True)` lanza una excepción si el dispositivo se quedó sin datos durante la escritura. Interceptando el camino real de Pipecat:

```
ruta ORIGINAL     · 310 writes    · underflows de PortAudio: 310   ← el 100%
ruta BUFFERIZADA  · 266 callbacks · underflows de PortAudio:   0
```

El arreglo es el estándar de audio en tiempo real: **desacoplar** la llegada de los datos de su reproducción con un *ring buffer* y un stream en modo **callback**. El dispositivo pide audio cuando lo necesita y se sirve del buffer; el productor solo deposita y nunca bloquea. Vive en [`salida_fluida.py`](salida_fluida.py) y los cuatro scripts de la lección ya lo usan.

Vale la pena entender por qué funciona, porque conecta con lo que medimos antes. Los deltas de audio llegan **a ráfagas** —medimos huecos de hasta 1,3 s entre uno y otro— pero el servidor entrega en total a ~1,9× tiempo real. Con un pre-buffer de 200 ms, el buffer acumula ventaja durante las ráfagas y la gasta en los huecos. Sin buffer, cada hueco es un silencio.

Puedes comprobarlo en tu máquina — reproduce la misma frase por las dos rutas y cuenta los underflows:

```bash
uv run python salida_fluida.py
```

> Esto **no es un defecto de Pipecat**: `LocalAudioTransport` está pensado para desarrollo, y en producción el audio no se reproduce en el servidor sino en el navegador o el teléfono, donde WebRTC ya trae su propio jitter buffer. La lección es más general y aplica a cualquier reproducción local: **nunca escribas audio de red directo al dispositivo.**

## Framework 3 · ElevenLabs Agents (plataforma)

Acá el cambio es de fondo, no de sintaxis: **el agente no vive en tu código**. Se crea como un recurso en la plataforma —prompt, idioma, voz, modelo, herramientas— y tu programa solo abre una conversación contra ese recurso y le presta el micrófono.

```python
creado = cliente.conversational_ai.agents.create(name="Luis", conversation_config={...})

conversacion = Conversation(
    cliente, creado.agent_id, requires_auth=True,
    audio_interface=DefaultAudioInterface(),   # micrófono y parlantes, resueltos
    client_tools=herramientas,
)
conversacion.start_session()
```

Lo que ganas es mucho: cero plomería de audio, turnos e interrupciones resueltos, telefonía incluida, y un panel con las conversaciones grabadas y transcritas — observabilidad que en los otros dos tienes que construir. El prompt se edita desde el dashboard sin volver a desplegar.

Lo que entregas también es mucho: la lógica conversacional queda en un proveedor. Migrarla después no es cambiar una línea, es un proyecto.

Dos detalles prácticos que descubrimos preparando la lección: un agente en español **debe** usar `eleven_flash_v2_5` o `turbo` (la API rechaza los demás modelos), y hay que usar una voz *premade*, porque las de la biblioteca —incluidas las de acento chileno— requieren plan pago.

El script crea el agente, conversa y lo **borra al salir**, para no dejar basura en la cuenta.

In [ ]:
if HAY_ELEVEN:
    print(correr("agente_elevenlabs.py", "--check"))
else:
    print("⚠️ Falta ELEVENLABS_API_KEY — se salta (llave gratis en elevenlabs.io).")

---

# Cómo funciona cada uno, por dentro

Hasta acá comparamos **criterios de decisión**. Eso sirve para elegir, pero no para entender — y sin entender la mecánica es imposible depurar, y muy fácil creer que los tres hacen lo mismo con distinta sintaxis. No lo hacen: operan en **alturas distintas**.

Hay una pregunta que separa a los cuatro mejor que cualquier otra: **¿contra qué vocabulario programas?** Es decir, ¿cuántos tipos de mensaje distintos tiene que conocer tu código para que esto funcione?

Contémoslos, en vez de suponerlos. Los tres paquetes están instalados en este entorno, así que los números salen de ahí.

In [ ]:
import dataclasses
import importlib
import inspect
import pkgutil
import typing

# Pipecat saluda con un banner por loguru al importarse; lo silenciamos para leer la tabla.
from loguru import logger

logger.remove()


def eventos_openai_crudo() -> set[str]:
    """Tipos de evento de la API Realtime, leídos de los modelos del SDK de OpenAI."""
    paquete = "openai.types.realtime"
    modulo = importlib.import_module(paquete)
    tipos: set[str] = set()
    for info in pkgutil.walk_packages(modulo.__path__, prefix=paquete + "."):
        try:
            submodulo = importlib.import_module(info.name)
        except Exception:
            continue
        for _, clase in inspect.getmembers(submodulo, inspect.isclass):
            campo = getattr(clase, "model_fields", {}).get("type")
            if campo is None:
                continue
            for literal in typing.get_args(campo.annotation):
                if isinstance(literal, str) and "." in literal:
                    tipos.add(literal)
    return tipos


# Los 11 que el cliente envía; el resto los emite el servidor.
EVENTOS_CLIENTE = {
    "session.update", "input_audio_buffer.append", "input_audio_buffer.commit",
    "input_audio_buffer.clear", "conversation.item.create", "conversation.item.delete",
    "conversation.item.truncate", "conversation.item.retrieve", "response.create",
    "response.cancel", "output_audio_buffer.clear",
}

crudos = eventos_openai_crudo()

# Agents SDK: sus eventos son dataclasses (no modelos pydantic), así que se leen distinto
from agents.realtime import events as eventos_sdk

sdk = set()
for _, clase in inspect.getmembers(eventos_sdk, inspect.isclass):
    if not dataclasses.is_dataclass(clase):
        continue
    try:
        pistas = typing.get_type_hints(clase)
    except Exception:
        continue
    for literal in typing.get_args(pistas.get("type")):
        if isinstance(literal, str):
            sdk.add(literal)
sdk = sorted(sdk)

# Pipecat: los frames, agrupados por familia (cada familia se planifica distinto)
import pipecat.frames.frames as frames_pc

familias: dict[str, list[str]] = {}
for nombre in dir(frames_pc):
    clase = getattr(frames_pc, nombre, None)
    if not inspect.isclass(clase) or not nombre.endswith("Frame"):
        continue
    familia = next(
        (b.__name__ for b in clase.__mro__[1:]
         if b.__name__ in ("SystemFrame", "DataFrame", "ControlFrame")),
        "base",
    )
    familias.setdefault(familia, []).append(nombre)

# ElevenLabs: los tipos que su SDK despacha (leídos del código del cliente)
from pathlib import Path
import re

fuente_el = Path(inspect.getfile(importlib.import_module(
    "elevenlabs.conversational_ai.conversation"))).read_text()
el_servidor = sorted(set(re.findall(r'== "([a-z_]+)"', fuente_el)))
el_cliente = sorted(set(re.findall(r'"type": "([a-z_]+)"', fuente_el)))

print(f"{'framework':<28} {'vocabulario':>12}   qué es")
print("-" * 78)
print(f"{'API Realtime a pelo':<28} {len(crudos):>12}   protocolo de cable "
      f"({len(EVENTOS_CLIENTE & crudos)} envías, {len(crudos - EVENTOS_CLIENTE)} recibes)")
print(f"{'OpenAI Agents SDK':<28} {len(sdk):>12}   eventos semánticos de sesión")
print(f"{'ElevenLabs Agents':<28} {len(el_servidor):>12}   eventos del servidor "
      f"(+{len(el_cliente)} mensajes que envías)")
total_pc = sum(len(v) for v in familias.values())
print(f"{'Pipecat':<28} {total_pc:>12}   tipos de frame del bus interno")
print()
for familia in ("SystemFrame", "DataFrame", "ControlFrame", "base"):
    print(f"    {familia:<14} {len(familias.get(familia, [])):>3} frames")

### Cómo leer esos números

No son comparables sin más, y ahí está justamente lo interesante.

**Los tres primeros son protocolos de cable**: mensajes que cruzan una red. Y forman una escala clara de cuánto te esconden:

- **56 eventos a pelo** → nadie decide nada por ti.
- **15 en el Agents SDK** → es una **reducción** de esos 56. El SDK ve los 56 y te entrega 15 con significado (y `raw_model_event` como escotilla para cuando necesites el evento crudo).
- **9 en ElevenLabs** → y de esos, solo **dos** te obligan a hacer algo (responder el `ping` y ejecutar el `client_tool_call`). El resto son avisos.

**El de Pipecat es otra cosa.** Esos ~130 frames no viajan por ninguna red: son el vocabulario de un **bus interno** en tu propio proceso. El número es grande porque Pipecat no te esconde su maquinaria, te la entrega. Es la diferencia entre aprender un protocolo y aprender un *sistema*.

Ahora veamos cada uno en detalle.

## Pipecat: todo es un frame en un bus

El modelo mental completo de Pipecat cabe en una frase: **la conversación es un flujo de frames tipados que atraviesa una lista de procesadores.** Todo lo demás se deduce de ahí.

### Las tres familias de frames (y por qué importan)

Los frames no son todos iguales — se **planifican** distinto, y esa es la parte que la gente no ve:

| Familia | Cola | Ejemplos | Para qué |
|---|---|---|---|
| **SystemFrame** | carril de **alta prioridad**, se salta la cola | `StartFrame`, `InputAudioRawFrame`, `UserStartedSpeakingFrame`, `InterruptionFrame`, `ErrorFrame` | Control urgente. La documentación dice que **no se ven afectados por las interrupciones** del usuario: son el mecanismo, no la carga. |
| **DataFrame** | cola normal, en orden | `TranscriptionFrame`, `LLMTextFrame`, `TTSAudioRawFrame`, `OutputAudioRawFrame` | La carga útil: el audio, el texto, la transcripción. |
| **ControlFrame** | cola normal, en orden con los datos | `TTSStartedFrame`/`TTSStoppedFrame`, `LLMFullResponseStartFrame`/`EndFrame`, `EndFrame` | Marcas de "empezó / terminó" que van *en secuencia* con los datos que delimitan. |

Que `InterruptionFrame` sea un `SystemFrame` y `TTSAudioRawFrame` un `DataFrame` no es taxonomía: es la razón por la que una interrupción puede adelantarse a 3 segundos de audio ya encolado.

### Dos direcciones

Los frames viajan `DOWNSTREAM` (hacia el final del pipeline, lo normal) o `UPSTREAM` (hacia atrás). Que exista la segunda dirección es lo que permite que una etapa tardía —el transporte, que es quien nota que el usuario empezó a hablar— le avise a las etapas anteriores.

### El ciclo de vida

1. **Arranque.** Un `StartFrame` recorre el pipeline procesador por procesador. Cada uno lo usa para inicializarse: abrir el WebSocket, cargar el modelo de VAD, reservar buffers. Si tu procesador necesita hablar con una API, es acá.
2. **Corriendo.** Frames de audio, texto y control fluyendo.
3. **Cierre.** Un `EndFrame` (o `CancelFrame` para cortar en seco) baja por el pipeline y cada etapa se apaga en orden.

### La regla que sorprende

> **Los procesadores no consumen los frames: los pasan.**

Un frame de transcripción no desaparece cuando el agregador de contexto lo usa — sigue viajando. Por eso puedes enchufar un procesador de logging, o uno que guarde la conversación, o uno que la mande a un dashboard, **sin tocar nada más**. Es la propiedad que hace a Pipecat extensible, y no es obvia viniendo de una arquitectura de llamadas a funciones.

### Un turno completo, frame por frame (modo cascada)

Esto es lo que realmente pasa entre que hablas y escuchas la respuesta:

| # | Etapa | Recibe | Emite |
|---|---|---|---|
| 1 | `transport.input()` | audio del micrófono | `InputAudioRawFrame` (y el VAD emite `UserStartedSpeakingFrame` → `UserStoppedSpeakingFrame`) |
| 2 | `stt` | `InputAudioRawFrame` | `InterimTranscriptionFrame`… → `TranscriptionFrame` (la final) |
| 3 | `user_aggregator` | `TranscriptionFrame` + las marcas de habla del VAD | arma el turno y emite el **contexto** completo |
| 4 | `llm` | el contexto | `LLMFullResponseStartFrame` → `LLMTextFrame`×N (token a token) → `LLMFullResponseEndFrame`. Si hay herramienta: `FunctionCallsStartedFrame` → `FunctionCallInProgressFrame` → `FunctionCallResultFrame` |
| 5 | `tts` | `LLMTextFrame` | `TTSStartedFrame` → `TTSAudioRawFrame`×N → `TTSStoppedFrame` |
| 6 | `transport.output()` | `TTSAudioRawFrame` | reproduce, y emite `BotStartedSpeakingFrame` → `BotStoppedSpeakingFrame` |
| 7 | `assistant_aggregator` | lo que dijo el bot | lo escribe de vuelta en el contexto, para el turno siguiente |

Fíjate en el paso 4: el LLM emite **un frame por token**, y el TTS los va consumiendo a medida que llegan. Eso es el streaming del que hablamos en la lección 3 — acá no hay que programarlo, es la forma del pipeline.

**En modo `realtime` desaparecen los pasos 2 y 5.** El servicio de LLM recibe `InputAudioRawFrame` y emite audio directamente. El `--check` de arriba lo imprime.

### La interrupción, que es el mejor detalle de diseño

Cuando el VAD detecta que el usuario empezó a hablar, alguien llama a `broadcast_interruption()`. Y esto es lo que hace (leído del código de `frame_processor.py`):

```python
async def broadcast_interruption(self):
    # docstring original: "Broadcast an InterruptionFrame both upstream and downstream."
    ...
    await self.broadcast_frame(InterruptionFrame)
```

Crea **dos** `InterruptionFrame` —cruzados por un `broadcast_sibling_id`— y empuja uno hacia adelante y otro hacia atrás. Cada procesador que lo recibe corre su `_start_interruption()`: el TTS bota el audio pendiente, el LLM cancela la generación, el transporte limpia su buffer de salida.

Por eso en Pipecat **interrumpir funciona sin que escribas nada**: no es una feature del servicio de voz, es una propiedad del bus. Y por eso el frame es `SystemFrame` — tiene que poder adelantarse a todo el audio ya encolado.

In [ ]:
# Los frames del turno de arriba, con su familia — para ver el patrón de planificación.
TURNO = [
    ("1. transport.input", ["InputAudioRawFrame", "UserStartedSpeakingFrame", "UserStoppedSpeakingFrame"]),
    ("2. stt",             ["InterimTranscriptionFrame", "TranscriptionFrame"]),
    ("3. user_aggregator", ["LLMContextFrame"]),
    ("4. llm",             ["LLMFullResponseStartFrame", "LLMTextFrame", "LLMFullResponseEndFrame",
                            "FunctionCallsStartedFrame", "FunctionCallResultFrame"]),
    ("5. tts",             ["TTSStartedFrame", "TTSAudioRawFrame", "TTSStoppedFrame"]),
    ("6. transport.output", ["OutputAudioRawFrame", "BotStartedSpeakingFrame", "BotStoppedSpeakingFrame"]),
    ("interrupción",       ["InterruptionFrame"]),
]

FAMILIA_DE = {nombre: familia for familia, nombres in familias.items() for nombre in nombres}

for etapa, nombres in TURNO:
    print(f"{etapa}")
    for nombre in nombres:
        familia = FAMILIA_DE.get(nombre, "¿NO EXISTE?")
        marca = {"SystemFrame": "⚡ prioridad", "DataFrame": "   datos",
                 "ControlFrame": "   control"}.get(familia, "   " + familia)
        print(f"      {nombre:<32} {marca}")
    print()

print("⚡ = SystemFrame: carril de alta prioridad, se adelanta al audio ya encolado.")
print("   Por eso InterruptionFrame puede cortar 3 segundos de TTS pendiente.")

## La API Realtime a pelo: 56 eventos en un WebSocket

Acá no hay bus ni pipeline: hay **una conexión** y un protocolo de mensajes JSON en las dos direcciones. Veamos los del servidor agrupados, que es donde se entiende la estructura.

In [ ]:
from collections import defaultdict

grupos: dict[str, list[str]] = defaultdict(list)
for evento in sorted(crudos - EVENTOS_CLIENTE):
    grupos[evento.split(".")[0]].append(evento)

print(f"EVENTOS DEL SERVIDOR ({len(crudos - EVENTOS_CLIENTE)}), por familia:\n")
for prefijo, lista in sorted(grupos.items(), key=lambda kv: -len(kv[1])):
    print(f"  {prefijo}  ({len(lista)})")
    for evento in lista:
        print(f"      {evento}")
    print()

print(f"EVENTOS QUE ENVÍA EL CLIENTE ({len(EVENTOS_CLIENTE & crudos)}):\n")
for evento in sorted(EVENTOS_CLIENTE & crudos):
    print(f"      {evento}")

### El turno, evento por evento

Con esos nombres a la vista, un turno completo se lee así:

```
  ── apertura, una sola vez ──────────────────────────────────
  ←  session.created                         el servidor saluda con su config por defecto
  →  session.update                          la tuya: instrucciones, voz, VAD, tools
  ←  session.updated                         confirmada

  ── el usuario habla ────────────────────────────────────────
  →  input_audio_buffer.append               tu audio, en base64, bloque a bloque (¡muchos!)
  ←  input_audio_buffer.speech_started       el VAD del servidor detectó voz
  ←  input_audio_buffer.speech_stopped       …y que terminó
  ←  input_audio_buffer.committed            el turno quedó cerrado
  ←  conversation.item.input_audio_transcription.completed    (opcional: qué oyó)

  ── el modelo responde ──────────────────────────────────────
  ←  response.created
  ←  response.output_audio.delta             ▓▓▓ el audio, en trozos — esto es lo que reproduces
  ←  response.output_audio_transcript.delta  el texto de lo que está diciendo (para tu UI/logs)
  ←  response.output_audio.done
  ←  response.done                           + uso de tokens
```

Con `turn_detection` configurado, el servidor decide solo cuándo empezó y terminó el turno y **crea la respuesta por su cuenta**. Sin `turn_detection`, tú tienes que mandar `input_audio_buffer.commit` y después `response.create` a mano.

### La herramienta: dos mensajes, y el segundo se olvida

```
  ←  response.function_call_arguments.delta   los argumentos, en streaming
  ←  response.function_call_arguments.done    { name, call_id, arguments }
  →  conversation.item.create                 { type: "function_call_output", call_id, output }
  →  response.create                          ← SIN ESTO EL AGENTE SE QUEDA MUDO
```

Ese segundo `response.create` es el error más común de la API. No hay mensaje de error: el agente simplemente deja de hablar, para siempre. Los tres frameworks lo mandan por ti.

### La interrupción: el problema que nadie anticipa

Con `interrupt_response: true` el servidor deja de generar cuando detecta que hablaste. Pero queda un problema que es **solo tuyo**, y es sutil:

> El servidor te mandó 8 segundos de audio. Tu parlante alcanzó a reproducir 3. El servidor cree que el usuario oyó los 8.

Si no arreglas eso, el contexto de la conversación contiene frases que el usuario nunca escuchó, y el modelo va a dar por sabido algo que nadie oyó. Para eso existen:

- **`conversation.item.truncate`** — le dices al servidor cuántos milisegundos se reprodujeron de verdad. El servidor **borra su transcripción** de la parte no oída, así el contexto coincide con la realidad.
- **`output_audio_buffer.clear`** — descarta el audio no reproducido (y también trunca).

Reconciliar "lo que mandé" con "lo que sonó" es, en esta arquitectura, responsabilidad de tu código.

### Los dos modos de detección de turno

| | `server_vad` | `semantic_vad` |
|---|---|---|
| **Cómo decide** | por **silencio**: `silence_duration_ms` | un **clasificador** sobre las palabras |
| **Falla cuando** | dudas a mitad de frase → te corta | — |
| **Ajuste** | umbral, padding, duración del silencio | `eagerness`: cuánto espera antes de asumir que terminaste |

Es el detalle que separa una demo de un producto: con `server_vad`, "quiero… un… taladro" se convierte en tres turnos.

## OpenAI Agents SDK: los 56 eventos, reducidos a 15

El SDK habla ese mismo protocolo, pero no te lo pasa. Consume los 56 y te entrega un flujo de eventos **con significado de agente**.

In [ ]:
SIGNIFICADO = {
    "agent_start": "el agente tomó el turno",
    "agent_end": "terminó de responder",
    "audio": "▓ trozo de audio para reproducir (response.output_audio.delta)",
    "audio_end": "no viene más audio de este turno",
    "audio_interrupted": "el usuario habló encima: bota lo que tengas en el parlante",
    "history_added": "se agregó un ítem a la conversación",
    "history_updated": "cambió el historial (acá vienen las transcripciones)",
    "tool_start": "va a ejecutar una herramienta",
    "tool_end": "la herramienta terminó (el ida y vuelta ya lo hizo el SDK)",
    "tool_approval_required": "espera tu OK antes de ejecutar (human in the loop)",
    "handoff": "pasó la conversación a otro agente",
    "guardrail_tripped": "un guardrail bloqueó la salida",
    "input_audio_timeout_triggered": "el usuario se quedó callado demasiado rato",
    "error": "algo falló",
    "raw_model_event": "← la escotilla: el evento crudo, tal como vino del cable",
}

print(f"EVENTOS DEL AGENTS SDK ({len(sdk)}):\n")
for evento in sdk:
    print(f"  {evento:<32} {SIGNIFICADO.get(evento, '')}")

print(f"\nReducción: {len(crudos)} eventos de cable → {len(sdk)} de agente "
      f"({len(crudos) - len(sdk)} absorbidos por el SDK).")

Mira la lista con cuidado, porque el valor está en lo que **no** aparece:

- No hay `input_audio_buffer.append`: llamas a `session.send_audio(bloque)`.
- No hay `conversation.item.create` + `response.create` tras una herramienta: decoras la función con `@function_tool` y recibes `tool_start` / `tool_end`. El ida y vuelta —incluido el `response.create` que todos olvidan— lo hace el SDK.
- No hay `conversation.item.truncate`: recibes `audio_interrupted` y llamas a `session.interrupt()`.
- Aparecen conceptos que **no existen en el protocolo**: `handoff`, `guardrail_tripped`, `tool_approval_required`. Son primitivos de *agentes*, no de voz — los mismos que ya usas en texto.

Y `raw_model_event` es la decisión de diseño más honesta del SDK: cuando la abstracción no te alcanza, te deja bajar al cable sin abandonar el framework.

Lo que sigue siendo tuyo: **el micrófono y el parlante**. `session.send_audio()` espera bytes PCM16 y `evento.audio.data` te devuelve bytes; de dónde salen y a dónde van lo cableas tú con PyAudio. Es la mitad de [`agente_openai.py`](agente_openai.py).

## ElevenLabs Agents: 9 eventos, y solo 2 te obligan a actuar

El vocabulario más chico de los cuatro, porque la plataforma no te está exponiendo un modelo: te está prestando un **servicio ya armado**.

### Los eventos que llegan

| Evento | Qué es | ¿Tienes que hacer algo? |
|---|---|---|
| `conversation_initiation_metadata` | primer mensaje: `conversation_id` y los formatos de audio negociados | no |
| `ping` | latido de la conexión | **sí** — responder `pong` o se cae (el SDK lo hace) |
| `audio` | ▓ audio en base64, con IDs de evento y alineación por carácter | reproducirlo |
| `agent_response` | el texto completo de lo que va a decir | no, es para tu UI |
| `user_transcript` | la transcripción final de lo que dijiste | no |
| `agent_response_correction` | ver abajo — el mejor detalle de la plataforma | no |
| `client_tool_call` | `{ tool_name, tool_call_id, parameters }` | **sí** — ejecutar y responder `client_tool_result` |
| `interruption` | el usuario habló encima | botar el audio en cola |
| `agent_chat_response_part` | texto en streaming (start / delta / stop), para modo chat | no |

Y los que tú envías, cuatro: `conversation_initiation_client_data` (config al abrir), `pong`, `client_tool_result`, y los bloques de audio.

### El orden de un turno

```
  ←  conversation_initiation_metadata
  ←  ping                    →  pong
  ←  audio                              el saludo inicial (first_message)
  ←  user_transcript                    lo que dijiste
  ←  agent_response  +  audio           lo que responde
  ←  client_tool_call        →  client_tool_result
  ←  audio                              la respuesta con el dato de la herramienta
  ←  agent_response_correction          si lo interrumpiste
```

### `agent_response_correction`: el problema de la interrupción, resuelto al revés

¿Te acuerdas del problema de la API a pelo — que el servidor cree que el usuario oyó 8 segundos cuando solo sonaron 3, y tú tienes que mandar `conversation.item.truncate` para reconciliar?

ElevenLabs resuelve **el mismo problema en la dirección opuesta**: cuando interrumpes, la plataforma te manda el texto **corregido** — lo que el agente alcanzó a decir de verdad, en vez de lo que pensaba decir. No tienes que calcular nada: te llega la respuesta.

Es el ejemplo más limpio de la diferencia de altitud entre los cuatro. Mismo problema físico —el audio en el buffer no es el audio que se oyó— y cuatro tratos distintos: a pelo lo reconcilias tú, el SDK te avisa, Pipecat lo resuelve difundiendo un frame por el bus, y ElevenLabs simplemente te dice el resultado.

## El mismo problema, a cuatro alturas

Tres cosas hay que resolver en **cualquier** agente de voz. Ver cómo las trata cada uno es la comparación que de verdad los distingue:

### 1. ¿De quién es el turno?

| | Cómo |
|---|---|
| **A pelo** | Configuras `turn_detection` y lees `input_audio_buffer.speech_started` / `.speech_stopped` |
| **Agents SDK** | La misma config, en `model_settings.audio.input.turn_detection` |
| **Pipecat** | Un **VAD local** (`SileroVADAnalyzer`, en tu máquina) que emite `UserStartedSpeakingFrame` |
| **ElevenLabs** | Lo decide la plataforma. Puedes mirar `vad_score` si lo habilitas |

La diferencia real: en Pipecat el VAD **corre en tu proceso** y no manda audio a nadie para detectar silencios. En los otros tres, decidir si estás hablando ya implica haber enviado tu audio.

### 2. El usuario interrumpió: ¿qué oyó realmente?

| | Cómo |
|---|---|
| **A pelo** | Tú reconcilias: `conversation.item.truncate` con los ms que de verdad se reprodujeron |
| **Agents SDK** | Evento `audio_interrupted` + `session.interrupt()` |
| **Pipecat** | `InterruptionFrame` difundido **arriba y abajo**; cada procesador se resetea solo |
| **ElevenLabs** | Evento `interruption` + `agent_response_correction` con el texto corregido |

### 3. El modelo quiere llamar una herramienta

| | Cómo |
|---|---|
| **A pelo** | `response.function_call_arguments.done` → `conversation.item.create` → **`response.create`** (olvidarlo = silencio) |
| **Agents SDK** | `@function_tool` sobre tu función; recibes `tool_start` / `tool_end` |
| **Pipecat** | `FunctionSchema` + `llm.register_function(...)`; el handler responde con `await params.result_callback(resultado)` |
| **ElevenLabs** | La herramienta se declara **en la config del agente**, en la plataforma; llega `client_tool_call` y respondes `client_tool_result` |

Fíjate en el último: es el único donde la herramienta **no está declarada en tu código**. La plataforma sabe que existe y decide cuándo llamarla; tu proceso solo la ejecuta. Eso es exactamente lo que significa "la lógica vive allá".

### Y una diferencia que no es de mecánica sino de forma

Los otros tres son **clientes**: abren una conexión, mandan y reciben mensajes. Pipecat es un **sistema de composición**: no te da un cliente, te da un bus donde el servicio de voz es *una etapa más*. Por eso es el único donde cambiar de proveedor —o meter un procesador propio en medio, o grabar la sesión, o correr dos modelos en paralelo— es una operación natural y no un parche.

## La tabla de decisión

|  | Nivel 0 (crudo) | OpenAI Agents SDK | Pipecat | ElevenLabs Agents |
|---|---|---|---|---|
| **Qué es** | protocolo de cable | cliente semántico | **sistema de composición** | servicio gestionado |
| **Vocabulario** | 56 eventos | 15 eventos | ~130 frames (bus interno) | 9 eventos (2 obligan a actuar) |
| **Licencia** | — | cerrado | **open source** (BSD) | cerrado |
| **Dónde vive la lógica** | tu código | tu código | tu código | **la plataforma** |
| **Dónde se declara la herramienta** | tu código | tu código (`@function_tool`) | tu código (`FunctionSchema`) | **la config del agente** |
| **Audio local resuelto** | no | no | **sí** | **sí** |
| **Detección de turno** | servidor | servidor | **VAD local** (Silero) | la plataforma |
| **Reconciliar la interrupción** | **tú** (`truncate`) | evento + `interrupt()` | frame difundido por el bus | te dan el texto corregido |
| **Cambiar de proveedor** | reescribir | atado a OpenAI | **una línea** | no aplica |
| **Telefonía / SIP** | tú | vía API | sí (Twilio, Daily) | **incluida** |
| **Observabilidad** | tú | traces del SDK | métricas + observers | **panel con grabaciones** |
| **Autohospedable** | sí (salvo el modelo) | no | **sí** | no |
| **Curva de aprendizaje** | alta | baja | media-alta | **muy baja** |
| **Líneas para este agente** | ~200 | ~180 | ~200 (dos arquitecturas) | ~140 |

### Cómo elegir, en una frase cada uno

- **ElevenLabs Agents** si el agente de voz *es* el producto y quieres estar en producción esta semana. Es el camino más corto a algo que funciona bien, y el más caro de abandonar.
- **Pipecat** si te importa no quedar amarrado, si necesitas mezclar proveedores, si tienes requisitos de datos que empujan a autohospedar, o si el agente de voz es una pieza de un sistema más grande que tú controlas.
- **OpenAI Agents SDK** si ya vives en OpenAI y tu agente de voz es la extensión de agentes de texto que ya tienes. La continuidad de primitivos vale mucho más de lo que parece.
- **Nivel 0** casi nunca en producción — pero léelo una vez, porque cuando algo falle en cualquiera de los tres, vas a estar depurando esto.

> Y el que no está en la tabla: **el sándwich de la lección 3**. Sigue siendo la respuesta correcta cuando la conversación no es en tiempo real (procesar grabaciones, un buzón de voz, un flujo asíncrono), cuando necesitas el texto como artefacto auditable de cada turno, o cuando quieres un modelo de razonamiento en el medio y puedes pagar la espera. Voice-to-voice no reemplaza al sándwich: resuelve el caso conversacional.

## Ahora háblales

Nada de lo anterior reemplaza la prueba real. Los cuatro scripts se corren desde la terminal, en esta carpeta, y se salen con Ctrl-C:

```bash
uv run python agente_openai.py                    # OpenAI Agents SDK
uv run python agente_pipecat.py                   # Pipecat, voice-to-voice
uv run python agente_pipecat.py --modo cascada    # Pipecat, el sándwich — compara al oído
uv run python agente_elevenlabs.py                # ElevenLabs Agents
uv run python nivel0_websocket_crudo.py           # sin framework
```

Tres cosas que vale la pena probar a propósito, porque son justo lo que el sándwich no podía:

1. **Interrúmpelo.** Empieza a hablar mientras Luis responde. Debería callarse en el acto.
2. **Duda a mitad de frase.** "Quiero… un… taladro." Un VAD por silencio te habría cortado; el semántico espera.
3. **Cambia el tono, no las palabras.** Pregunta lo mismo apurado y después relajado, o molesto. En el sándwich el agente veía el mismo texto en los dos casos. Acá no.

Y compara al oído `--modo cascada` contra `--modo realtime` de Pipecat: es el mismo agente, la misma herramienta y el mismo código, con dos elementos de diferencia en una lista.

## Qué nos llevamos

- El salto de sándwich a **voice-to-voice** no es una optimización, es un cambio de arquitectura: se elimina la barrera de "esperar el texto completo" y se deja de tirar a la basura el tono del usuario.
- La métrica correcta en voz no es el tiempo total sino el **tiempo hasta la primera sílaba**, y ahí la diferencia que medimos es de varias veces — con el agravante de que la medición le regaló al sándwich la transcripción.
- Los cuatro programas resuelven el mismo caso, pero **no operan a la misma altura**, y eso se ve en el vocabulario contra el que programas: 56 eventos de cable a pelo, 15 semánticos en el Agents SDK, 9 en ElevenLabs, y ~130 frames en Pipecat — que es grande porque no es un protocolo sino un **sistema** que te entregan por dentro.
- Los tres problemas que **todo** agente de voz tiene que resolver —de quién es el turno, qué oyó realmente el usuario al interrumpir, y el ida y vuelta de la herramienta— aparecen en los cuatro, y comparar *cómo* los tratan distingue mejor que cualquier tabla de features.
- Pipecat es el único que no es un **cliente** sino un **compositor**: el servicio de voz es una etapa más del bus. De ahí sale todo lo demás — cambiar de proveedor, VAD local, meter un procesador propio en medio.
- Ninguno resuelve todo: el SDK de OpenAI no toca tu micrófono, Pipecat te cobra en conceptos, ElevenLabs te cobra en dependencia (y es el único donde la herramienta se declara fuera de tu código).
- Y una lección de plomería que no es glamorosa pero arruina cualquier demo: **nunca escribas audio de red directo al dispositivo**. Los deltas llegan a ráfagas; sin un jitter buffer el parlante se queda seco entre trozo y trozo. Medimos underflow en el 100% de los writes en la ruta local de Pipecat, y 0 con un ring buffer.
- Y el sándwich **no murió**: sigue ganando en lo asíncrono, en lo auditable y cuando necesitas razonamiento en el medio.

Con esto cierra la clase 5.3: la señal (L1), la síntesis (L2), la arquitectura en cascada (L3) y los modelos que oyen y hablan directo (L4).